# Therapy AI Fine-Tuning with Mistral-7B

This notebook fine-tunes Mistral-7B-Instruct-v0.2 for university student mental health support.

## Features:
- 🧠 **111 therapy conversations** for comprehensive training
- 🎯 **University-specific scenarios** (academic stress, financial pressure, social challenges)
- 🚨 **Crisis intervention protocols** (suicidal ideation, self-harm detection)
- 📊 **PHQ-9 & GAD-7 assessment integration**
- 💙 **Empathetic, culturally sensitive responses**

## Requirements:
- Kaggle GPU (P100) - 30 hours/week free
- ~8GB RAM
- ~15GB storage for model and dataset


## ⚠️ IMPORTANT: Execution Order

**Please run the cells in this exact order:**

1. **Cell 2**: Install packages (run this first!)
2. **Cell 4**: Import libraries and setup
3. **Cell 6**: Load dataset
4. **Cell 8**: Configure model
5. **Cell 10**: Configure LoRA
6. **Cell 12**: Configure training
7. **Cell 14**: Start training
8. **Cell 16**: Save and test model

**If you get import errors, restart the kernel and run Cell 2 first!**


## Step 1: Install Required Libraries


In [ ]:
# Install required packages
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install -q transformers datasets accelerate peft bitsandbytes trl
%pip install -q huggingface_hub

print("✅ All packages installed successfully!")


## Step 2: Import Libraries and Setup


In [12]:
# Check if packages are installed, if not, install them
try:
    import torch
    print("✅ PyTorch already installed")
except ImportError:
    print("❌ PyTorch not found, installing...")
    %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

try:
    from datasets import load_dataset
    print("✅ Datasets already installed")
except ImportError:
    print("❌ Datasets not found, installing...")
    %pip install -q datasets

try:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
    print("✅ Transformers already installed")
except ImportError:
    print("❌ Transformers not found, installing...")
    %pip install -q transformers

try:
    from peft import LoraConfig, PeftModel
    print("✅ PEFT already installed")
except ImportError:
    print("❌ PEFT not found, installing...")
    %pip install -q peft

try:
    from trl import SFTTrainer
    print("✅ TRL already installed")
except ImportError:
    print("❌ TRL not found, installing...")
    %pip install -q trl

# Now import everything
import torch
import json
import os
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
import warnings
warnings.filterwarnings("ignore")

# Check GPU availability
print(f"🚀 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📱 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

print("✅ Setup complete!")


✅ PyTorch already installed
✅ Datasets already installed
✅ Transformers already installed
✅ PEFT already installed
✅ TRL already installed
🚀 CUDA available: False
✅ Setup complete!


## Step 3: Load and Prepare Dataset


In [13]:
# Load the therapy dataset from your local file
# Check if running on Kaggle or locally
import os
if os.path.exists('/kaggle/input/therapy-dataset/therapy_fine_tuning_dataset.jsonl'):
    # Running on Kaggle
    dataset_path = '/kaggle/input/therapy-dataset/therapy_fine_tuning_dataset.jsonl'
    print("🔍 Running on Kaggle - using Kaggle dataset path")
else:
    # Running locally
    dataset_path = 'therapy_fine_tuning_dataset.jsonl'
    print("🔍 Running locally - using local dataset path")
    
    # Check if local file exists
    if not os.path.exists(dataset_path):
        print(f"❌ Dataset file not found: {dataset_path}")
        print("📁 Please make sure 'therapy_fine_tuning_dataset.jsonl' is in the same directory as this notebook")
        raise FileNotFoundError(f"Dataset file not found: {dataset_path}")

dataset = load_dataset('json', data_files=dataset_path, split='train')
print(f"📊 Loaded dataset: {len(dataset)} conversations")

# Format conversations for training
def format_conversation(example):
    """Convert messages format to text format for training"""
    messages = example['messages']
    text = ""
    for message in messages:
        if message['role'] == 'user':
            text += f"<s>[INST] {message['content']} [/INST] "
        elif message['role'] == 'model':
            text += f"{message['content']} </s>"
    return {"text": text}

# Apply formatting
dataset = dataset.map(format_conversation)
print("✅ Dataset formatted for training!")
print(f"📝 Sample formatted text: {dataset[0]['text'][:200]}...")


🔍 Running locally - using local dataset path


Generating train split: 110 examples [00:00, 6254.89 examples/s]


📊 Loaded dataset: 110 conversations


Map: 100%|██████████| 110/110 [00:00<00:00, 1812.81 examples/s]

✅ Dataset formatted for training!
📝 Sample formatted text: <s>[INST] Hello, I'm feeling really stressed about my studies [/INST] Hello! I'm here to listen and support you. I understand that academic stress can be incredibly overwhelming, especially when you'r...


## Step 4: Configure Model and Tokenizer


In [17]:
# Import required modules (in case they're not available from previous cell)
import torch
from transformers import BitsAndBytesConfig
print("✅ Imports confirmed")

# Model configuration
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
new_model_name = "therapy-ai-mistral-local"  # Changed for local use

# Configure 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("🔄 Loading model and tokenizer...")

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✅ Model and tokenizer loaded successfully!")
print(f"🧠 Model: {model_name}")
print(f"📝 Tokenizer vocab size: {len(tokenizer)}")


✅ Imports confirmed


NameError: name 'torch' is not defined

## Step 5: Configure LoRA for Efficient Fine-Tuning


In [ ]:
# Import required modules
from peft import LoraConfig
print("✅ PEFT imports confirmed")

# Configure LoRA (Low-Rank Adaptation)
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

print("✅ LoRA configuration ready!")
print(f"🎯 Target modules: {peft_config.target_modules}")
print(f"📊 LoRA rank: {peft_config.r}")


✅ LoRA configuration ready!
🎯 Target modules: {'up_proj', 'down_proj', 'gate_proj', 'v_proj', 'q_proj', 'o_proj', 'k_proj'}
📊 LoRA rank: 64


## Step 6: Configure Training Arguments


In [ ]:
# Import required modules
from transformers import TrainingArguments
print("✅ TrainingArguments import confirmed")

# Training arguments optimized for Kaggle P100 GPU
training_args = TrainingArguments(
    output_dir="./therapy-ai-results",
    num_train_epochs=3,
    per_device_train_batch_size=1,  # Small batch for P100
    gradient_accumulation_steps=4,  # Effective batch size = 4
    optim="paged_adamw_32bit",
    logging_steps=10,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,  # Better stability than fp16
    save_steps=100,
    save_total_limit=2,
    remove_unused_columns=False,
    warmup_steps=100,
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    report_to="none",  # Disable wandb/tensorboard
    dataloader_pin_memory=False,  # Reduce memory usage
)

print("✅ Training arguments configured!")
print(f"🎯 Epochs: {training_args.num_train_epochs}")
print(f"📊 Batch size: {training_args.per_device_train_batch_size}")
print(f"📈 Learning rate: {training_args.learning_rate}")


Can not specify world size due to 
TrainingArguments requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.
. Turn average_tokens_across_devices to False.


✅ Training arguments configured!
🎯 Epochs: 3
📊 Batch size: 1
📈 Learning rate: 0.0002


## Step 7: Initialize Trainer and Start Training


In [ ]:
# Import required modules
from trl import SFTTrainer
print("✅ SFTTrainer import confirmed")

# Initialize SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    tokenizer=tokenizer,
    args=training_args,
    formatting_func=format_conversation,
    max_seq_length=512,
    packing=False,
)

print("✅ Trainer initialized successfully!")
print(f"📊 Training samples: {len(dataset)}")
print(f"📝 Max sequence length: 512")

print("\n🚀 Starting fine-tuning...")
print("⏱️ This may take 2-3 hours on Kaggle P100 GPU")

# Start training
trainer.train()

print("✅ Training completed successfully!")


NameError: name 'model' is not defined

## Step 8: Save and Test the Fine-Tuned Model


In [ ]:
# Import required modules
import torch
print("✅ Torch import confirmed for testing")

# Save the fine-tuned model
trainer.save_model(new_model_name)
tokenizer.save_pretrained(new_model_name)

print(f"✅ Model saved as: {new_model_name}")

# Test the fine-tuned model
def test_model(prompt, max_length=200):
    # Format the prompt for Mistral
    formatted_prompt = f"<s>[INST] You are a compassionate therapy assistant. {prompt} [/INST]"
    
    # Tokenize
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # Decode response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract just the AI response
    ai_response = response.split("[/INST]")[-1].strip()
    
    return ai_response

# Test with sample prompts
test_prompts = [
    "I'm feeling really stressed about my studies",
    "I'm having a panic attack right now",
    "I feel depressed and hopeless"
]

print("\n🧪 Testing the fine-tuned model...")
print("=" * 50)

for prompt in test_prompts:
    print(f"\n👤 User: {prompt}")
    response = test_model(prompt)
    print(f"🤖 AI: {response}")
    print("-" * 30)

print("\n🎉 Fine-tuning and testing completed successfully!")


NameError: name 'trainer' is not defined